# FE BAR Bronze Silver AutoLoader Pipeline - Build Evidence

**Pipeline:** FE BAR Bronze Silver AutoLoader Pipeline  
**Pipeline ID:** c0db28b8-dbd2-432d-99cf-4ea2aea60274  
**Catalog:** workshop_guy_catalog  
**Schema:** fe_bar_a0  
**Workspace:** fevm-workshop-guy  
**Date:** 2026-10-14  
**Genie Conversation Thread ID:** 2172058462486781  
**Branch:** feature/bronze-silver-autoloader-pipeline  

---

## Architecture

**Bronze Layer** (Streaming Tables via AutoLoader):
- `dim_customer_bronze` ← `/Volumes/.../staging/customers/`
- `dim_merchant_bronze` ← `/Volumes/.../staging/merchants/`
- `fact_transactions_bronze` ← `/Volumes/.../staging/transactions/`
- `fact_prior_fraud_cases_bronze` ← `/Volumes/.../staging/prior_fraud_cases/`

**Silver Layer** (Materialized Views with dedup + expectations):
- `dim_customer_silver` — column mask on `avg_monthly_spend` (PII group)
- `dim_merchant_silver`
- `fact_transactions_silver` — row filter on `amount > 10000` (PII group)
- `fact_prior_fraud_cases_silver`

In [0]:
%sql
-- BRONZE LAYER: Row counts for all ingested tables
SELECT 'dim_customer_bronze' AS table_name, COUNT(*) AS row_count FROM workshop_guy_catalog.fe_bar_a0.dim_customer_bronze
UNION ALL
SELECT 'dim_merchant_bronze', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_bronze
UNION ALL
SELECT 'fact_transactions_bronze', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_bronze
UNION ALL
SELECT 'fact_prior_fraud_cases_bronze', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_bronze
ORDER BY table_name;

**Executed 2026-10-14 | Pipeline update ID: d7163364-af6a-423d-a7f5-db29d0472a49**

| table_name | row_count |
|---|---|
| dim_customer_bronze | 10,768 |
| dim_merchant_bronze | 5,402 |
| fact_prior_fraud_cases_bronze | 53,788 |
| fact_transactions_bronze | 537,505 |

All 4 bronze streaming tables ingested successfully via AutoLoader.

In [0]:
%sql
-- SILVER LAYER: Row counts for deduplicated tables
SELECT 'dim_customer_silver' AS table_name, COUNT(*) AS row_count FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver
UNION ALL
SELECT 'dim_merchant_silver', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_silver
UNION ALL
SELECT 'fact_transactions_silver', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_prior_fraud_cases_silver', COUNT(*) FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver
ORDER BY table_name;

**Executed 2026-10-14**

| table_name | row_count |
|---|---|
| dim_customer_silver | 9,289 |
| dim_merchant_silver | 4,666 |
| fact_prior_fraud_cases_silver | 46,409 |
| fact_transactions_silver | 463,881 |

Deduplication reduced row counts across all tables:
- Customers: 10,768 → 9,289 (1,479 duplicates removed)
- Merchants: 5,402 → 4,666 (736 duplicates removed)
- Transactions: 537,505 → 463,881 (73,624 duplicates removed)
- Prior Fraud Cases: 53,788 → 46,409 (7,379 duplicates removed)

In [0]:
%sql
-- QUALITY CHECK: Verify zero NULLs in business key columns after expectations applied
SELECT 'dim_customer_silver' AS table_name, 'customer_id' AS key_col,
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_count
FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver
UNION ALL
SELECT 'dim_merchant_silver', 'merchant_id',
       SUM(CASE WHEN merchant_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_silver
UNION ALL
SELECT 'fact_transactions_silver', 'txn_id',
       SUM(CASE WHEN txn_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_transactions_silver', 'customer_id',
       SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_prior_fraud_cases_silver', 'case_id',
       SUM(CASE WHEN case_id IS NULL THEN 1 ELSE 0 END)
FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver
ORDER BY table_name, key_col;

**PASS: All null_count values are 0**

| table_name | key_col | null_count |
|---|---|---|
| dim_customer_silver | customer_id | 0 |
| dim_merchant_silver | merchant_id | 0 |
| fact_prior_fraud_cases_silver | case_id | 0 |
| fact_transactions_silver | customer_id | 0 |
| fact_transactions_silver | txn_id | 0 |

Expectations with `ON VIOLATION DROP ROW` successfully removed all NULL business keys.

In [0]:
%sql
-- DEDUP CHECK: Verify no duplicate business keys in silver tables
SELECT 'dim_customer_silver' AS table_name, 'customer_id' AS key_col,
       COUNT(*) - COUNT(DISTINCT customer_id) AS duplicate_count
FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver
UNION ALL
SELECT 'dim_merchant_silver', 'merchant_id',
       COUNT(*) - COUNT(DISTINCT merchant_id)
FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_silver
UNION ALL
SELECT 'fact_transactions_silver', 'txn_id',
       COUNT(*) - COUNT(DISTINCT txn_id)
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver
UNION ALL
SELECT 'fact_prior_fraud_cases_silver', 'case_id',
       COUNT(*) - COUNT(DISTINCT case_id)
FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver
ORDER BY table_name;

**PASS: All duplicate_count values are 0**

| table_name | key_col | duplicate_count |
|---|---|---|
| dim_customer_silver | customer_id | 0 |
| dim_merchant_silver | merchant_id | 0 |
| fact_prior_fraud_cases_silver | case_id | 0 |
| fact_transactions_silver | txn_id | 0 |

ROW_NUMBER() deduplication confirmed: zero duplicate business keys remain in any silver table.

In [0]:
%sql
-- COLUMN MASK: avg_monthly_spend is masked for non-PII users
-- If current user is NOT in PII group, avg_monthly_spend will show NULL
SELECT * FROM workshop_guy_catalog.fe_bar_a0.dim_customer_silver LIMIT 10;

**Column mask on `avg_monthly_spend` is ACTIVE.**

Current user is NOT in the `pii` group, so `avg_monthly_spend` shows NULL:

| customer_id | home_region | risk_segment | tenure | avg_monthly_spend |
|---|---|---|---|---|
| CUST-00001 | EU | medium | 45 | NULL |
| CUST-00002 | EU | low | 43 | NULL |
| CUST-00003 | EU | low | 25 | NULL |
| CUST-00004 | US-East | low | 33 | NULL |
| CUST-00005 | APAC | low | 33 | NULL |

Mask UDF: `workshop_guy_catalog.fe_bar_a0.mask_avg_monthly_spend`
Logic: `IF(IS_ACCOUNT_GROUP_MEMBER('pii'), avg_monthly_spend, NULL)`

In [0]:
%sql
SELECT * FROM workshop_guy_catalog.fe_bar_a0.dim_merchant_silver LIMIT 10;

In [0]:
%sql
-- ROW FILTER: Rows with amount > 10000 are hidden for non-PII users
-- Check max amount visible to current user
SELECT MAX(amount) AS max_visible_amount,
       COUNT(*) AS visible_row_count,
       SUM(CASE WHEN amount > 10000 THEN 1 ELSE 0 END) AS rows_over_10k
FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver;

**Row filter on `amount > 10000` is ACTIVE.**

Current user is NOT in the `pii` group:

| max_visible_amount | visible_row_count | rows_over_10k |
|---|---|---|
| 9,995.99 | 463,881 | 0 |

Max visible amount is $9,995.99 — all transactions over $10,000 are hidden.
PII group members would see the full dataset including high-value transactions.

Filter UDF: `workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns`
Logic: `IF(IS_ACCOUNT_GROUP_MEMBER('pii'), TRUE, amount <= 10000)`

| txn_id | customer_id | merchant_id | amount | channel | device_id | geo_distance | is_fraud | timestamp |
|---|---|---|---|---|---|---|---|---|
| TXN-0000003 | CUST-09665 | MERCH-02604 | 36.05 | card_present | DEV-IQCAF-096641 | 1247.9 | false | 2026-06-16T14:37:12Z |
| TXN-0000006 | CUST-06629 | MERCH-01365 | 780.91 | card_present | DEV-MCVIT-066283 | 601.26 | false | 2026-10-01T11:24:06Z |
| TXN-0000010 | CUST-08193 | MERCH-00608 | 74.56 | card_present | DEV-GBDLS-081921 | 1140.38 | false | 2026-05-17T06:05:03Z |
| TXN-0000011 | CUST-08144 | MERCH-01630 | 38.68 | card_present | DEV-XTGFC-081433 | 496.79 | false | 2026-10-08T08:17:20Z |
| TXN-0000012 | CUST-09912 | MERCH-04106 | 64.51 | card_present | DEV-WVFNM-099112 | 763.81 | false | 2026-05-21T06:53:01Z |

```
Function:      workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns
Type:          SCALAR
Input:         amount DOUBLE
Returns:       BOOLEAN
Deterministic: true
Data Access:   CONTAINS SQL
Body:          IF(IS_ACCOUNT_GROUP_MEMBER('pii'), TRUE, amount <= 10000)
```

```
Function:      workshop_guy_catalog.fe_bar_a0.mask_avg_monthly_spend
Type:          SCALAR
Input:         avg_monthly_spend DOUBLE
Returns:       DOUBLE
Deterministic: true
Data Access:   CONTAINS SQL
Body:          IF(IS_ACCOUNT_GROUP_MEMBER('pii'), avg_monthly_spend, NULL)
```

In [0]:
%sql
SELECT * FROM workshop_guy_catalog.fe_bar_a0.fact_transactions_silver LIMIT 10;

In [0]:
%sql
SELECT * FROM workshop_guy_catalog.fe_bar_a0.fact_prior_fraud_cases_silver LIMIT 10;

In [0]:
%sql
-- Show the row filter function definition
DESCRIBE FUNCTION EXTENDED workshop_guy_catalog.fe_bar_a0.filter_high_amount_txns;

In [0]:
%sql
-- Show the column mask function definition
DESCRIBE FUNCTION EXTENDED workshop_guy_catalog.fe_bar_a0.mask_avg_monthly_spend;